<a href="https://colab.research.google.com/github/c5063565-coder/dnnls_final_project/blob/main/Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as FT
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import math
import os
import time
import json
import textwrap
import re

from bs4 import BeautifulSoup
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer
from google.colab import drive
import gc

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
drive.mount('/content/gdrive')

BASE_FOLDER = '/content/gdrive/MyDrive/DNN/DL_Checkpoints'

FOLDERS = {
    'base':BASE_FOLDER,
    'checkpoints': os.path.join(BASE_FOLDER, 'checkpoints'),
    'baseline':os.path.join(BASE_FOLDER, 'experiments', 'baseline'),
    'exp1':os.path.join(BASE_FOLDER, 'experiments', 'exp1_transformer_resnet'),
    'exp2':os.path.join(BASE_FOLDER, 'experiments', 'exp2_contrastive'),
}

for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)

print("Drive folder structure:")
for name, path in FOLDERS.items():
    print(f"{name:12} = {path}")

def save_checkpoint(model, optimizer, epoch, loss, filename):
    path = os.path.join(FOLDERS['checkpoints'], filename)
    torch.save({
        'epoch':epoch,
        'model_state_dict':model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict() if optimizer else None,
        'loss':loss,
    }, path)
    print(f"Checkpoint saved: checkpoints/{filename}")

def load_checkpoint(model, optimizer=None, filename="checkpoint.pth"):
    path = os.path.join(FOLDERS['checkpoints'], filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    ckpt = torch.load(path, map_location=torch.device('cpu'))
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer and ckpt.get('optimizer_state_dict'):
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"Loaded: {filename}  (epoch {ckpt.get('epoch', 0)})")
    return model, optimizer, ckpt.get('epoch', 0), ckpt.get('loss', None)

def save_log(lines, experiment):
    filename = f"training_log_{experiment}.txt"
    path= os.path.join(FOLDERS[experiment], filename)
    content= '\n'.join(lines)
    with open(path, 'w') as f:
        f.write(content)
    with open(filename, 'w') as f:
        f.write(content)

    print(f"Log saved → experiments/{experiment}/{filename}")
def save_figure(fig, filename, experiment):
    path = os.path.join(FOLDERS[experiment], filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f"Figure saved → experiments/{experiment}/{filename}")
print("\nAll utilities ready.")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
gc.collect()

EMB_DIM = 16
LATENT_DIM = 16
NUM_LAYERS = 1
DROPOUT = 0.1
N_EPOCHS = 15
BATCH_SIZE = 8
LR = 0.001
MAX_LEN = 120
IMAGE_HW= (60, 125)

USE_TRANSFORMER_ENCODER = False
USE_RESNET_ENCODER = False
USE_CONTRASTIVE_ROI = False
USE_FRAME_AWARE_GROUNDING = False
USE_ENTITY_POOLING = False
USE_COT_TEXT = False

LAMBDA_CONTRAST = 0.10
LAMBDA_GROUND_MSE = 0.10
LAMBDA_REID = 0.10
LAMBDA_ENTITY_POOL = 0.05
CONTRASTIVE_TAU = 0.07

EXP_NAME = "{}_{}_{}" .format(
    "transformer" if USE_TRANSFORMER_ENCODER else "lstm",
    "resnet" if USE_RESNET_ENCODER else "cnn",
    "contrast" if USE_CONTRASTIVE_ROI else "nocontrast"
)

if not USE_TRANSFORMER_ENCODER and not USE_RESNET_ENCODER:
    EXP_FOLDER = 'baseline'
elif USE_TRANSFORMER_ENCODER and USE_RESNET_ENCODER and not USE_CONTRASTIVE_ROI:
    EXP_FOLDER = 'exp1'
elif USE_TRANSFORMER_ENCODER and USE_RESNET_ENCODER and USE_CONTRASTIVE_ROI:
    EXP_FOLDER = 'exp2'
else:
    EXP_FOLDER = 'baseline'  # fallback

print(f"Device:{device}")
print(f"Experiment:{EXP_NAME}")
print(f"Folder:experiments/{EXP_FOLDER}")
print(f"Transformer:{USE_TRANSFORMER_ENCODER}")
print(f"ResNet:{USE_RESNET_ENCODER}")
print(f"Contrastive:{USE_CONTRASTIVE_ROI}")

In [ ]:
train_dataset = load_dataset("daniel3303/StoryReasoning", split="train")
test_dataset = load_dataset("daniel3303/StoryReasoning", split="test")

print(f"Train samples : {len(train_dataset)}")
print(f"Test samples : {len(test_dataset)}")
print(f"\nColumns in dataset: {train_dataset.column_names}")

sample = train_dataset[0]
print(f"\nSample keys : {list(sample.keys())}")
print(f"Number of frames : {sample['frame_count']}")
print(f"Number of images : {len(sample['images'])}")
print(f"Story text preview: {sample['story'][:200]}")
print(f"CoT preview : {str(sample.get('chain_of_thought', ''))[:200]}")


In [ ]:
sample = train_dataset[0]
print(f"Story text preview: {sample['story'][:200]}")

In [ ]:
def parse_gdi_text(text):
    soup = BeautifulSoup(text, 'html.parser')
    images = []

    for gdi in soup.find_all('gdi'):
        image_id = None
        if gdi.attrs:
            for attr_name in gdi.attrs:
                if 'image' in attr_name.lower():
                    image_id = attr_name.replace('image', '')
                    break
        if not image_id:
            match = re.search(r'<gdi\s+image(\d+)', str(gdi))
            if match:
                image_id = match.group(1)
        if not image_id:
            image_id = str(len(images) + 1)

        images.append({
            'image_id':image_id,
            'description': gdi.get_text().strip(),
            'objects':[o.get_text().strip() for o in gdi.find_all('gdo')],
            'actions':[a.get_text().strip() for a in gdi.find_all('gda')],
            'locations':[l.get_text().strip() for l in gdi.find_all('gdl')],
        })

    return images
def show_image(ax, image):
    ax.imshow(image.permute(1, 2, 0).cpu().numpy().clip(0, 1))
def _parse_markdown_table(block):
    lines = [l.rstrip() for l in block.splitlines()]
    table_lines = [l for l in lines if l.strip().startswith("|")]

    if len(table_lines) < 3:
        return []

    headers = [h.strip() for h in table_lines[0].strip("|").split("|")]
    rows= []

    for line in table_lines[2:]:
        cols = [c.strip() for c in line.strip("|").split("|")]
        if len(cols) == len(headers):
            rows.append(dict(zip(headers, cols)))

    return rows
def parse_cot_grounding(chain_of_thought):
    frames = {}
    pattern = re.compile(r"^##\s*Image\s+(\d+)", flags=re.MULTILINE)
    matches = list(pattern.finditer(chain_of_thought or ""))

    for i, m in enumerate(matches):
        idx = int(m.group(1)) - 1
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(chain_of_thought)
        section = chain_of_thought[start:end]

        frames[idx] = {"characters": [], "objects": []}

        for tag, key in [("Characters", "characters"), ("Objects", "objects")]:
            hit = re.search(
                rf"###\s*{tag}(.*?)(?=\n###|\n##|$)",
                section, re.DOTALL
            )
            if hit:
                id_key = "Character ID" if key == "characters" else "Object ID"
                for row in _parse_markdown_table(hit.group(1)):
                    eid = row.get(id_key, "").strip()
                    bbox = row.get("Bounding Box", "").strip()
                    if eid and bbox:
                        try:
                            x1, y1, x2, y2 = [int(v) for v in bbox.split(",")]
                            frames[idx][key].append({
                                "id":   eid,
                                "bbox": [x1, y1, x2, y2]
                            })
                        except:
                            pass

    return frames
def crop_and_resize(pil_img, bbox, out_hw=IMAGE_HW):
    W, H = pil_img.size
    x1, y1, x2, y2 = bbox

    x1, x2 = max(0, x1), min(W-1, x2)
    y1, y2 = max(0, y1), min(H-1, y2)

    if x2 <= x1: x2 = min(W-1, x1+1)
    if y2 <= y1: y2 = min(H-1, y1+1)

    crop = pil_img.crop((x1, y1, x2, y2))
    return transforms.Compose([
        transforms.Resize(out_hw),
        transforms.ToTensor()
    ])(crop)
def pick_reid_pair(frames_cot):
    import random
    id_to_dets = {}

    for f_idx, content in frames_cot.items():
        for det in content.get("characters", []) + content.get("objects", []):
            eid  = det.get("id")
            bbox = det.get("bbox")
            if eid and bbox:
                id_to_dets.setdefault(eid, []).append((f_idx, bbox))

    candidates = [e for e, d in id_to_dets.items() if len(d) >= 2]

    if not candidates:
        return None

    eid = random.choice(candidates)
    (f1, b1), (f2, b2) = random.sample(id_to_dets[eid], 2)
    return f1, f2, b1, b2, eid
def extract_cot_text_for_frame(cot, frame_idx, max_chars=600):
    if not cot:
        return ""

    pattern = re.compile(r"^##\s*Image\s+(\d+)", flags=re.MULTILINE)
    matches = list(pattern.finditer(cot))

    for i, m in enumerate(matches):
        if int(m.group(1)) - 1 == frame_idx:
            start = m.end()
            end = matches[i+1].start() if i+1 < len(matches) else len(cot)
            target = cot[start:end]

            lines = [
                l for l in target.splitlines()
                if not l.strip().startswith("|")
                and set(l.strip()) > set("-|:")
            ]
            text = " ".join(l.strip() for l in lines if l.strip())
            return re.sub(r"\s+", " ", text).strip()[:max_chars]

    return ""

sample = train_dataset[0]
parsed = parse_gdi_text(sample['story'])
cot_frames = parse_cot_grounding(sample.get('chain_of_thought', ''))

print(f"Frames parsed from story: {len(parsed)}")
print(f"Frame 0 description: {parsed[0]['description'][:80]}")
print(f"Frame 0 objects: {parsed[0]['objects']}")
print(f"Frame 0 actions: {parsed[0]['actions']}")
print(f"Frame 0 locations: {parsed[0]['locations']}")
print(f"CoT frames with bboxes: {list(cot_frames.keys())}")
print("\nHelper functions ready.")

In [ ]:
class SequencePredictionDataset(Dataset):
    def __init__(self, dataset, tokenizer, K=4, max_len=MAX_LEN, image_hw=IMAGE_HW):
        self.dataset = dataset
        self.tokenizer= tokenizer
        self.K = K
        self.max_len = max_len
        self.transform = transforms.Compose([
            transforms.Resize(image_hw),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        story = self.dataset[idx]
        frames = story["images"]
        image_attrs = parse_gdi_text(story["story"])
        cot= story.get("chain_of_thought", "")
        cot_frames = parse_cot_grounding(cot)
        frame_tensors = []
        description_list = []

        for fi in range(self.K):
            img = self.transform(FT.equalize(frames[fi]))
            frame_tensors.append(img)
            desc = image_attrs[fi]["description"]
            if USE_COT_TEXT:
                cot_txt = extract_cot_text_for_frame(cot, fi)
                if cot_txt:
                    desc = desc + " [COT] " + cot_txt
            ids = self.tokenizer(
                desc,
                return_tensors = "pt",
                padding = "max_length",
                truncation = True,
                max_length = self.max_len
            ).input_ids.squeeze(0)
            description_list.append(ids)
        image_target = self.transform(FT.equalize(frames[self.K]))
        target_desc  = image_attrs[self.K]["description"]
        target_ids   = self.tokenizer(
            target_desc,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = self.max_len
        ).input_ids
        roi1 = torch.zeros(3, *IMAGE_HW)
        roi2 = torch.zeros(3, *IMAGE_HW)
        roi_valid = 0
        roi_frame_idx = -1
        ent_id = ""

        pair = pick_reid_pair(cot_frames)
        if pair:
            f1, f2, b1, b2, eid = pair
            if f1 < self.K and f2 < self.K:
                try:
                    roi1 = crop_and_resize(frames[f1], b1, IMAGE_HW)
                    roi2 = crop_and_resize(frames[f2], b2, IMAGE_HW)
                    roi_valid = 1
                    roi_frame_idx = f1
                    ent_id = eid
                except:
                    pass
        return (
            torch.stack(frame_tensors),
            torch.stack(description_list),
            image_target,
            target_ids,
            roi1,
            roi2,
            torch.tensor(roi_valid),
            torch.tensor(roi_frame_idx),
            ent_id
        )

class TextTaskDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        attrs = parse_gdi_text(self.dataset[idx]["story"])
        fi= np.random.randint(0, 5)  # pick a random frame
        return attrs[fi]["description"] # return just the text string

class AutoEncoderTaskDataset(Dataset):
    def __init__(self, dataset):
        self.dataset= dataset
        self.transform = transforms.Compose([
            transforms.Resize(IMAGE_HW),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        frames = self.dataset[idx]["images"]
        fi = torch.randint(0, 5, (1,)).item()   # random frame index
        return (self.transform(frames[fi]),)          # tuple with one image tensor

print("Testing dataset classes...")

tokenizer_test = BertTokenizer.from_pretrained(
    "google-bert/bert-base-uncased", padding=True, truncation=True)
sp_test_ds = SequencePredictionDataset(train_dataset, tokenizer_test)

frames, descs, img_tgt, txt_tgt, roi1, roi2, roi_valid, roi_frame, ent_id = sp_test_ds[0]

print(f"\nSequencePredictionDataset:")
print(f"  Context frames shape : {frames.shape}")
print(f"  Descriptions shape: {descs.shape}")
print(f"  Target image shape: {img_tgt.shape}")
print(f"  Target text shape: {txt_tgt.shape}")
print(f"  ROI 1 shape: {roi1.shape}")
print(f"  ROI valid: {roi_valid.item()}")
print(f"  Entity ID: {ent_id}")

txt_ds   = TextTaskDataset(train_dataset)
sample_t = txt_ds[0]
print(f"\nTextTaskDataset:")
print(f"  Sample text: {sample_t[:80]}")

ae_ds     = AutoEncoderTaskDataset(train_dataset)
(sample_i,) = ae_ds[0]
print(f"\nAutoEncoderTaskDataset:")
print(f"  Image shape: {sample_i.shape}")

print("\nAll dataset classes ready.")

In [ ]:
class EncoderLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            input_size = emb_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0
        )

    def forward(self, x):
        emb = self.embedding(x)
        out, (h, c) = self.lstm(emb)
        return out, h, c
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe= torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)
class EncoderTransformer(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim,
                 num_layers=2, nhead=4, dropout=0.1, max_len=512):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.input_proj = (
            nn.Linear(emb_dim, hidden_dim)
            if emb_dim != hidden_dim
            else nn.Identity()
        )
        self.pos_enc = PositionalEncoding(hidden_dim, max_len, dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model= hidden_dim,
            nhead = nhead,
            dim_feedforward = hidden_dim * 4,
            dropout = dropout,
            batch_first = True,
            norm_first = True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, x):
        pad_mask = (x == 0)
        emb = self.embedding(x)
        proj = self.input_proj(emb)
        x = self.pos_enc(proj)
        out = self.transformer(
            x,
            src_key_padding_mask = pad_mask
        )
        out = self.out_proj(out)
        mask_f = (~pad_mask).unsqueeze(-1).float()
        pooled = (out * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        h = pooled.unsqueeze(0)
        c = torch.zeros_like(h)
        return out, h, c
    def get_attention_weights(self, x, max_tokens=20):
        x = x[:, :max_tokens]
        pad_mask = (x == 0)
        emb = self.embedding(x)
        proj = self.input_proj(emb)
        enc = self.pos_enc(proj)
        layer = self.transformer.layers[0]
        with torch.no_grad():
            _, attn_weights = layer.self_attn(
                enc, enc, enc,
                key_padding_mask = pad_mask,
                need_weights = True,
                average_attn_weights = False
            )
        return attn_weights
class DecoderLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            input_size = emb_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0
        )
        self.out = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x, h, c):
        emb = self.embedding(x)
        out, (h, c) = self.lstm(emb, (h, c))
        predictions = self.out(out)
        return predictions, h, c
class Seq2SeqLSTM(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        _, h, c = self.encoder(src)
        preds, _, _ = self.decoder(tgt[:, :-1], h, c)
        return preds
print("Testing text encoder models...")
vocab_size = tokenizer_test.vocab_size
test_input = torch.randint(0, vocab_size, (2, MAX_LEN))

lstm_enc = EncoderLSTM(vocab_size, EMB_DIM, LATENT_DIM)
out, h, c = lstm_enc(test_input)
print(f"\nLSTM Encoder:")
print(f"  Output shape : {out.shape}")
print(f"  Hidden shape : {h.shape}")
print(f"  Cell shape : {c.shape}")

tfm_enc = EncoderTransformer(vocab_size, EMB_DIM, LATENT_DIM, num_layers=2, nhead=4)
out, h, c = tfm_enc(test_input)
print(f"\nTransformer Encoder:")
print(f"  Output shape : {out.shape}")
print(f"  Hidden shape: {h.shape}")
print(f"  Cell shape: {c.shape}")

lstm_dec = DecoderLSTM(vocab_size, EMB_DIM, LATENT_DIM)
preds, h2, c2 = lstm_dec(test_input, h, c)
print(f"\nLSTM Decoder:")
print(f"  Predictions shape : {preds.shape}")

seq2seq = Seq2SeqLSTM(tfm_enc, lstm_dec)
output  = seq2seq(test_input, test_input)
print(f"\nSeq2Seq (Transformer encoder + LSTM decoder):")
print(f"  Output shape : {output.shape}")
print("\nText encoder models ready.")

In [ ]:
class Backbone(nn.Module):
    def __init__(self, latent_dim=16, output_w=8, output_h=16):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 16, 7, stride=2, padding=3),
            nn.GroupNorm(8, 16),
            nn.LeakyReLU(0.1),

            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.GroupNorm(8, 32),
            nn.LeakyReLU(0.1),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.LeakyReLU(0.1),
        )
        self.flatten_dim = 64 * output_w * output_h
        self.fc1 = nn.Sequential(
            nn.Linear(self.flatten_dim, latent_dim),
            nn.ReLU()
        )
    def forward(self, x):
        x = self.encoder_conv(x)
        x = x.view(-1, self.flatten_dim)
        return self.fc1(x)
class ResNetBackbone(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )
    def forward(self, x):
        features = self.features(x)
        return self.projection(features)
class VisualEncoder(nn.Module):
    def __init__(self, latent_dim=16, output_w=8, output_h=16):
        super().__init__()
        if USE_RESNET_ENCODER:
            self.content_backbone = ResNetBackbone(latent_dim)
            self.context_backbone = ResNetBackbone(latent_dim)
            print("  VisualEncoder: using ResNet-18 backbone")
        else:
            self.content_backbone = Backbone(latent_dim, output_w, output_h)
            self.context_backbone = Backbone(latent_dim, output_w, output_h)
            print("  VisualEncoder: using CNN baseline backbone")

        self.projection = nn.Linear(2 * latent_dim, latent_dim)
    def forward(self, x):
        z_content = self.content_backbone(x)
        z_context = self.context_backbone(x)
        z = torch.cat([z_content, z_context], dim=1)
        return self.projection(z)
class VisualDecoder(nn.Module):
    def __init__(self, latent_dim=16, output_w=8, output_h=16):
        super().__init__()
        self.flatten_dim = 64 * output_w * output_h
        self.output_w = output_w
        self.output_h = output_h
        self.imh = 60
        self.imw = 125
        self.fc1 = nn.Linear(latent_dim, self.flatten_dim)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=(1,1)),
            nn.GroupNorm(8, 32),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose2d(32, 16, 5, stride=2, padding=2, output_padding=1),
            nn.GroupNorm(8, 16),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose2d(16, 3, 7, stride=2, padding=3, output_padding=(1,1)),
            nn.Sigmoid()
        )
    def forward(self, z):
        x = self.fc1(z)
        x_content = self.decode_image(x)
        x_context = self.decode_image(x)
        return x_content, x_context
    def decode_image(self, x):
        x = x.view(-1, 64, self.output_w, self.output_h)
        x = self.decoder_conv(x)
        return x[:, :, :self.imh, :self.imw]
class VisualAutoencoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = VisualEncoder(latent_dim)
        self.decoder = VisualDecoder(latent_dim)
    def forward(self, x):
        z = self.encoder(x)
        x_c, x_ctx = self.decoder(z)
        return x_c, x_ctx
print("Testing visual encoder models...\n")
test_img = torch.randn(2, 3, 60, 125)

cnn_bb = Backbone(latent_dim=LATENT_DIM)
z_cnn = cnn_bb(test_img)
print(f"CNN Backbone:")
print(f"Input: {test_img.shape}")
print(f"Output : {z_cnn.shape}")

res_bb = ResNetBackbone(latent_dim=LATENT_DIM)
z_res = res_bb(test_img)
print(f"\nResNet Backbone:")
print(f"Input: {test_img.shape}")
print(f"Output : {z_res.shape}")

vis_enc = VisualEncoder(latent_dim=LATENT_DIM)
z_enc = vis_enc(test_img)
print(f"\nVisualEncoder:")
print(f"Output : {z_enc.shape}")

vis_dec = VisualDecoder(latent_dim=LATENT_DIM)
x_c, x_ctx = vis_dec(z_enc)
print(f"\nVisualDecoder:")
print(f"Content shape : {x_c.shape}")
print(f"Context shape : {x_ctx.shape}")

vis_ae = VisualAutoencoder(latent_dim=LATENT_DIM)
x_c, x_ctx = vis_ae(test_img)
print(f"\nVisualAutoencoder:")
print(f"Reconstructed content : {x_c.shape}")
print(f"Reconstructed context : {x_ctx.shape}")
print("\nVisual encoder models ready.")

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn    = nn.Linear(hidden_dim, 1)
        self.softmax = nn.Softmax(dim=1)
    def forward(self, rnn_out):
        scores  = self.attn(rnn_out)
        scores  = scores.squeeze(2)
        weights = self.softmax(scores)
        context = torch.bmm(
            weights.unsqueeze(1),
            rnn_out
        ).squeeze(1)
        return context
class SequencePredictor(nn.Module):
    def __init__(self, visual_ae, text_ae, latent_dim, gru_hidden):
        super().__init__()
        self.image_encoder = visual_ae.encoder
        self.text_encoder  = text_ae.encoder
        fusion_dim = latent_dim * 2
        self.temporal_rnn  = nn.GRU(
            input_size  = fusion_dim,
            hidden_size = latent_dim,
            batch_first = True
        )
        self.attention = Attention(gru_hidden)
        self.projection = nn.Sequential(
            nn.Linear(gru_hidden * 2, latent_dim),
            nn.ReLU()
        )
        self.image_decoder = visual_ae.decoder
        self.text_decoder  = text_ae.decoder
        self.fused_to_h0 = nn.Linear(latent_dim, latent_dim)
        self.fused_to_c0 = nn.Linear(latent_dim, latent_dim)

    def forward(self, image_seq, text_seq, target_seq):
        B, S, C, H, W = image_seq.shape
        img_flat = image_seq.view(B*S, C, H, W)
        txt_flat = text_seq.view(B*S, -1)
        z_v_flat = self.image_encoder(img_flat)
        _, h_t, c_t = self.text_encoder(txt_flat)
        z_t_flat    = h_t.squeeze(0)
        z_v_seq = z_v_flat.view(B, S, -1)
        z_t_seq = z_t_flat.view(B, S, -1)
        fusion = torch.cat([z_v_seq, z_t_seq], dim=2)
        gru_out, _ = self.temporal_rnn(fusion)
        context = self.attention(gru_out)
        h_last = gru_out[:, -1, :]
        fused = self.projection(
            torch.cat([h_last, context], dim=1)
        )
        pred_img_content, pred_img_context = self.image_decoder(fused)
        h0 = self.fused_to_h0(fused).unsqueeze(0)
        c0 = self.fused_to_c0(fused).unsqueeze(0)
        pred_text, _, _ = self.text_decoder(
            target_seq.squeeze(1)[:, :-1], h0, c0
        )
        return (
            pred_img_content,
            pred_img_context,
            pred_text,
            h0,
            c0,
            z_v_seq,
            z_t_seq
        )
print("Testing SequencePredictor.\n")
_vis_ae  = VisualAutoencoder(latent_dim=LATENT_DIM)
_enc     = (EncoderTransformer(tokenizer_test.vocab_size, EMB_DIM, LATENT_DIM)
            if USE_TRANSFORMER_ENCODER
            else EncoderLSTM(tokenizer_test.vocab_size, EMB_DIM, LATENT_DIM))
_dec     = DecoderLSTM(tokenizer_test.vocab_size, EMB_DIM, LATENT_DIM)
_txt_ae  = Seq2SeqLSTM(_enc, _dec)
seq_pred = SequencePredictor(_vis_ae, _txt_ae, LATENT_DIM, LATENT_DIM)
B = 2
test_images = torch.randn(B, 4, 3, 60, 125)
test_texts = torch.randint(0, tokenizer_test.vocab_size,
                             (B, 4, MAX_LEN))
test_target = torch.randint(0, tokenizer_test.vocab_size,
                             (B, 1, MAX_LEN))
pred_img, pred_ctx, pred_text, h0, c0, z_v, z_t = seq_pred(
    test_images, test_texts, test_target
)
print(f"Inputs:")
print(f"Image sequence : {test_images.shape}")
print(f"Text sequence : {test_texts.shape}")
print(f"Target text : {test_target.shape}")
print(f"\nOutputs:")
print(f"Predicted image : {pred_img.shape}")
print(f"Predicted context : {pred_ctx.shape}")
print(f"Predicted text : {pred_text.shape}")
print(f"Image embeddings : {z_v.shape}")
print(f"Text embeddings : {z_t.shape}")
total_params = sum(p.numel() for p in seq_pred.parameters())
print(f"\nTotal parameters: {total_params:,}")
print("\nSequencePredictor ready.")

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
print("=" * 55)
print("Building text autoencoder")
print("=" * 55)
if USE_TRANSFORMER_ENCODER:
    print("  Encoder : Transformer (2 layers, 4 heads)")
    encoder = EncoderTransformer(
        vocab_size= tokenizer_test.vocab_size,
        emb_dim = EMB_DIM,
        hidden_dim = LATENT_DIM,
        num_layers = 2,
        nhead= 4,
        dropout= DROPOUT
    ).to(device)

    decoder = DecoderLSTM(
        vocab_size = tokenizer_test.vocab_size,
        emb_dim    = EMB_DIM,
        hidden_dim = LATENT_DIM,
        num_layers = NUM_LAYERS,
        dropout    = DROPOUT
    ).to(device)

    text_ae = Seq2SeqLSTM(encoder, decoder).to(device)
    print("  Transformer encoder initialised — will pretrain in.")

else:
    print(" Encoder : LSTM baseline")
    encoder = EncoderLSTM(
        vocab_size = tokenizer_test.vocab_size,
        emb_dim    = EMB_DIM,
        hidden_dim = LATENT_DIM,
        num_layers = NUM_LAYERS,
        dropout    = DROPOUT
    ).to(device)

    decoder = DecoderLSTM(
        vocab_size = tokenizer_test.vocab_size,
        emb_dim = EMB_DIM,
        hidden_dim = LATENT_DIM,
        num_layers = NUM_LAYERS,
        dropout = DROPOUT
    ).to(device)

    text_ae = Seq2SeqLSTM(encoder, decoder).to(device)
    try:
        pth_path = os.path.join(FOLDERS['checkpoints'], 'text_autoencoder.pth')
        if os.path.exists(pth_path):
            state = torch.load(pth_path, map_location=device)
            if isinstance(state, dict) and 'model_state_dict' in state:
                text_ae.load_state_dict(state['model_state_dict'])
            else:
                text_ae.load_state_dict(state)
            print("Loaded pretrained LSTM weights from text_autoencoder.pth")
        else:
            print("text_autoencoder.pth not found in Drive.")
            print("Upload it to: MyDrive/DNN/DL_Checkpoints/checkpoints/")
            print("Training LSTM from scratch instead.")
            text_ae.apply(init_weights)
    except Exception as e:
        print(f"Could not load weights: {e}")
        print("Training from scratch.")
        text_ae.apply(init_weights)
for p in text_ae.parameters():
    p.requires_grad = False
txt_total = sum(p.numel() for p in text_ae.parameters())
print(f"  Total parameters : {txt_total:,}  (frozen until pretraining)")
print()
print("=" * 55)
print("Building visual autoencoder")
print("=" * 55)
if USE_RESNET_ENCODER:
    print("  Encoder : ResNet-18 (ImageNet pretrained)")
else:
    print("  Encoder : CNN baseline (random init)")
visual_ae = VisualAutoencoder(latent_dim=LATENT_DIM).to(device)
if not USE_RESNET_ENCODER:
    visual_ae.apply(init_weights)
    print("  Kaiming Normal init applied to CNN.")
else:
    visual_ae.encoder.content_backbone.projection.apply(init_weights)
    visual_ae.encoder.context_backbone.projection.apply(init_weights)
    visual_ae.decoder.apply(init_weights)
    print("  Projection + decoder initialised. ResNet weights preserved.")
vis_total = sum(p.numel() for p in visual_ae.parameters())
vis_train = sum(p.numel() for p in visual_ae.parameters() if p.requires_grad)
print(f"Total parameters     : {vis_total:,}")
print(f"Trainable parameters : {vis_train:,}")
print("=" * 55)
print("Building SequencePredictor")
print("=" * 55)
seq_pred = SequencePredictor(
    visual_ae  = visual_ae,
    text_ae    = text_ae,
    latent_dim = LATENT_DIM,
    gru_hidden = LATENT_DIM
).to(device)
seq_total = sum(p.numel() for p in seq_pred.parameters())
seq_train = sum(p.numel() for p in seq_pred.parameters() if p.requires_grad)
print(f"Total parameters     : {seq_total:,}")
print(f"Trainable parameters : {seq_train:,}")
print(f"(text_ae frozen - trainable count is lower than total)")
print("=" * 55)
print("MODEL SUMMARY")
print("=" * 55)
print(f"Experiment: {EXP_NAME}")
print(f"Text encoder: {'Transformer' if USE_TRANSFORMER_ENCODER else 'LSTM'}")
print(f"Visual encoder: {'ResNet-18' if USE_RESNET_ENCODER else 'CNN'}")
print(f"Contrastive loss : {'ON' if USE_CONTRASTIVE_ROI else 'OFF'}")
print(f"Device: {device}")
print(f"  {'Component':<25} {'Params':>12}")
print(f"  {'-'*38}")
print(f"  {'Text Autoencoder':<25} {txt_total:>12,}")
print(f"  {'Visual Autoencoder':<25} {vis_total:>12,}")
print(f"  {'Full SequencePredictor':<25} {seq_total:>12,}")
print("=" * 55)
print("Running forward pass test")
seq_pred.eval()
with torch.no_grad():
    dummy_imgs = torch.randn(2, 4, 3, 60, 125).to(device)
    dummy_txt  = torch.randint(0, tokenizer_test.vocab_size, (2, 4, MAX_LEN)).to(device)
    dummy_tgt  = torch.randint(0, tokenizer_test.vocab_size, (2, 1, MAX_LEN)).to(device)
    pred_img, pred_ctx, pred_text, h0, c0, z_v, z_t = seq_pred(
        dummy_imgs, dummy_txt, dummy_tgt
    )
print(f"Predicted image : {pred_img.shape}")
print(f"Predicted text  : {pred_text.shape}")
print(f"Image embeddings : {z_v.shape}")
print(f"Text embeddings : {z_t.shape}")
print(" All models ready. Proceed to pretraining")